# GraphRAG Pipeline Schritt für Schritt
Dieses Notebook führt durch jeden Abschnitt der modularen Pipeline: Query Planner, Cypher-Retriever, Antwortgenerator und der abschließende End-to-End-Lauf.

## Voraussetzungen
- Neo4j-Instanz mit geladenem IoT-Graphen läuft und ist via Bolt erreichbar.
- Die Python-Abhängigkeiten aus `requirements.txt` sind installiert.
- Optional: Setze `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `GRAPH_LLM_MODEL` und `GRAPH_LLM_TEMPERATURE` als Umgebungsvariablen.
- Führe die folgenden Zellen nacheinander aus, um jeden Pipeline-Schritt getrennt zu inspizieren.

In [1]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
SRC_PATH = PROJECT_ROOT / 'src'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://neo4j:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')
LLM_MODEL = os.getenv('GRAPH_LLM_MODEL', 'ollama/llama3')
TEMPERATURE = float(os.getenv('GRAPH_LLM_TEMPERATURE', '0.0'))

print(f'Using Neo4j at {NEO4J_URI} as {NEO4J_USER}')
print(f'LLM model: {LLM_MODEL} (temperature={TEMPERATURE})')

Using Neo4j at neo4j://neo4j:7687 as neo4j
LLM model: ollama/llama3 (temperature=0.0)


## Komponenten laden
Wir importieren die modularen Bausteine und instanziieren sie einmalig für die folgenden Schritte.

In [ ]:
from src.graph.pipeline import build_pipeline

# Pipeline einmalig instanziieren
graph_pipeline = build_pipeline(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
)

print("Komponenten bereit und Pipeline initialisiert.")

Komponenten bereit und Pipeline initialisiert.


### Schritt 1 – Query Planner
Wir schicken eine Beispiel-Frage durch den Query-Planner und extrahieren die erzeugte Cypher-Query.

In [ ]:
sample_question = "Welche Temperatur wurde heute um 12:00 Uhr von Sensor 5 gemessen?"

# Schritt 1: Nur Planner-Ausgabe inspizieren
result_step1 = graph_pipeline.run(graph_pipeline,data={"question": sample_question})
planner_outputs = result_step1["planner"]
cypher_query = planner_outputs["cypher"]
raw_output = planner_outputs["raw_output"]

print("Planner-Rohtext:\n", raw_output)
print("\nExtrahierte Cypher-Query:\n", cypher_query)

AttributeError: 'NoneType' object has no attribute 'run'

### Schritt 2 – Cypher Retriever
Wir führen die Cypher-Query gegen Neo4j aus und inspizieren die zurückgegebenen Dokumente.

In [ ]:
# Schritt 2: Retriever-Dokumente inspizieren
retrieval_result = result_step1["retriever"]
documents = retrieval_result["documents"]
print(f"\n{len(documents)} Dokument(e) vom Graphen erhalten.")

if not documents:
    raise ValueError("Der Retriever hat keine Dokumente geliefert – prüfe die Cypher-Query oder den Graph-Inhalt.")

for idx, doc in enumerate(documents[:5], start=1):
    snippet = doc.content
    if len(snippet) > 300:
        snippet = snippet[:300] + "…"
    print(f"\nDokument {idx:02d}:\n{snippet}")

### Schritt 3 – Generator
Wir übergeben die Kontext-Dokumente an den LLM und erhalten eine finale Antwort.

In [ ]:
# Schritt 3: Generierte Antwort anzeigen
generation_result = result_step1["generator"]
answer = generation_result["answer"]
print("Antwort der Pipeline:\n", answer)

### Schritt 4 – Gesamtlauf (optional)
Zum Abschluss lassen wir die komplette Pipeline laufen, um alle Schritte automatisch hintereinander auszuführen.

In [ ]:
def run_full_pipeline(question: str):
    result = graph_pipeline.run(data={"question": question})
    answer = result["generator"]["answer"]
    docs = result["retriever"]["documents"]
    print("Antwort:", answer)
    print(f"\n{len(docs)} Kontext-Dokument(e) zurückgegeben.")
    return result

full_response = run_full_pipeline(sample_question)